# 감정 모델 추가 학습 (KEMDy19·KEMDy20 일부)
`HyunaZ/hubert_emotion`을 KEMDy 데이터로 이어서 학습합니다. **런타임 → 런타임 유형 변경 → GPU(T4)** 로 먼저 바꾸세요.

데이터는 공개 저장소 `SeolRoh/Multi-Still_ETRI`(ETRI 대회 코드)에 들어 있는 KEMDy19·KEMDy20 일부(세션 1~3에 라벨, 라벨 발화 약 2,400개)를 씁니다. 따로 내려받거나 Drive에 올릴 필요가 없습니다.
위에서부터 한 칸씩 실행하면 됩니다.

> 주의: KEMDy 원본은 ETRI 사용 조건 동의가 필요한 데이터입니다. 연구·시연 목적으로만 쓰고, 서비스로 이어갈 때는 ETRI 라이선스를 확인하세요.

In [ ]:
# 1) GPU 확인 — 'Tesla T4' 같은 표가 나와야 합니다
!nvidia-smi

In [ ]:
# 2) 코드 받기 (공개 저장소)
!git clone https://github.com/jaeuk6797-cell/kuccton.git
%cd kuccton
!git checkout feat/finetune-kemdy19

In [ ]:
# 3) 라이브러리 설치 (torch는 Colab에 이미 있습니다)
!pip -q install transformers pytorch-lightning librosa huggingface_hub

In [ ]:
# 4) 데이터 받기 (약 1GB, 몇 분 걸립니다)
!git clone --depth 1 https://github.com/SeolRoh/Multi-Still_ETRI.git /content/KEMDy
!ls /content/KEMDy

In [ ]:
# 5) 학습 전에 데이터 확인 — 라벨이 붙은 발화 수(약 2,400개)와 라벨별 개수가 나옵니다
#    라벨 읽은 방식이 '두 줄 헤더: Total Evaluation/Emotion'이면 정상입니다
!python training/finetune.py --data /content/KEMDy --inspect

In [ ]:
# 6) 학습 (T4 기준 에폭당 몇 분). '[시작 전]'과 '[에폭 N]'의 macro-F1을 비교하세요
#    데이터가 적어서 에폭을 늘려도 크게 오르지 않을 수 있습니다
!python training/finetune.py --data /content/KEMDy --out /content/out --epochs 4

In [ ]:
# 7) 결과를 Drive에 저장 (런타임이 끊겨도 남도록)
from google.colab import drive
drive.mount('/content/drive')
!cp /content/out/hubert_emotion_kemdy19.ckpt /content/drive/MyDrive/hubert_emotion_kemdy19.ckpt
!ls -la /content/drive/MyDrive/hubert_emotion_kemdy19.ckpt

## 끝난 뒤
1. Drive의 `hubert_emotion_kemdy19.ckpt`를 PC로 내려받습니다.
2. 서버를 켤 때 `EMOTION_CKPT` 환경변수로 그 파일 경로를 알려 주면 새 모델을 씁니다.
   `set EMOTION_CKPT=C:\경로\hubert_emotion_kemdy19.ckpt` 후 `python -m uvicorn server:app --port 8000` (server 폴더에서)
3. **개선됐는지는 '[시작 전]'보다 '[에폭 N]'의 macro-F1이 오르는지로 판단하세요.** 오르지 않으면 쓰지 마세요.